# H121 IV/TC Skill Figures

Rerunnable figure package for the full-span H121 CDR ASCAT step-4 Instrument Variable (IV) and Triple Collocation (TC) output bundle.

Map conventions for this decision package:

- maps are clipped to latitude >= -60 degrees;
- maps use Robinson projection when Cartopy is available;
- spatial means use GEOSldas tile-area weights, never plain tile counts.

Outputs are written to `projects/ascat_da/output/h121_iv_tc_skill_figures/` as PNG plus CSV summary tables.



In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from IPython.display import display

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception as exc:
    cfeature = None
    HAS_CARTOPY = False
    warnings.warn(f"Cartopy unavailable; map cells will fall back to lon/lat axes: {exc}")


mpl.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})

ROOT = Path.cwd()
while ROOT.name != "geosldas-analysis" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if ROOT.name != "geosldas-analysis":
    raise RuntimeError("Run from inside the geosldas-analysis repository")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from projects.iv_tc.iv_tc.readers import read_tilecoord

BUNDLE = ROOT / "data" / "step4_h121_cdr_test_20260725"
TILECOORD = ROOT / "projects" / "obs_scaling_params" / "test_data" / "inputs" / "OLv7_M36_MULTI_type_13_H121.ldas_tilecoord.bin"
OUT = ROOT / "projects" / "ascat_da" / "output" / "h121_iv_tc_skill_figures"
OUT.mkdir(parents=True, exist_ok=True)

MAP_LAT_MIN = -60.0
MAP_EXTENT = [-180.0, 180.0, MAP_LAT_MIN, 90.0]
LAND_FACE = "0.92"
ZERO_NEUTRAL_FRACTION = 0.02
M36_NX = 964

RUN_ORDER = ["OL", "DA_legacy", "DA_H121", "DA_SMAP_comb_fp_scaled"]
RUN_LABELS = {
    "OL": "OL",
    "DA_legacy": "DA legacy",
    "DA_H121": "DA H121",
    "DA_SMAP_comb_fp_scaled": "DA SMAP",
}
SENSOR_ORDER = ["smap_l3", "smosic", "cygnss_l3", "ascat_h121", "ascat_h119_h120"]
SENSOR_LABELS = {
    "smap_l3": "SMAP L3",
    "smosic": "SMOS-IC",
    "cygnss_l3": "CYGNSS L3",
    "ascat_h121": "ASCAT H121",
    "ascat_h119_h120": "ASCAT H119/H120",
}
TC_COMBO_LABELS = {
    "smap_l3__model__smosic": "SMAP L3 + model + SMOS-IC",
    "cygnss_l3__model__smosic": "CYGNSS L3 + model + SMOS-IC",
    "smosic__model__ascat_h121": "SMOS-IC + model + ASCAT H121",
}

if not BUNDLE.exists():
    raise FileNotFoundError(BUNDLE)
if not TILECOORD.exists():
    raise FileNotFoundError(TILECOORD)





## Shared Helpers



In [ ]:
def save_fig(fig, name: str) -> None:
    png = OUT / f"{name}.png"
    fig.savefig(png, bbox_inches="tight", pad_inches=0.04)
    print(f"saved {png.relative_to(ROOT)}")
    display(fig)

def weighted_mean(values, weights):
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    ok = np.isfinite(v) & np.isfinite(w) & (w > 0)
    if not ok.any():
        return np.nan
    return float(np.sum(v[ok] * w[ok]) / np.sum(w[ok]))


def weighted_median(values, weights):
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    ok = np.isfinite(v) & np.isfinite(w) & (w > 0)
    if not ok.any():
        return np.nan
    v = v[ok]
    w = w[ok]
    order = np.argsort(v)
    v = v[order]
    w = w[order]
    cdf = np.cumsum(w) / np.sum(w)
    return float(v[np.searchsorted(cdf, 0.5)])


def area_summary(df: pd.DataFrame, value_col: str) -> dict:
    ok = (
        np.isfinite(df[value_col].to_numpy(dtype=float))
        & np.isfinite(df["area"].to_numpy(dtype=float))
        & (df["area"].to_numpy(dtype=float) > 0)
        & (df["lat"].to_numpy(dtype=float) >= MAP_LAT_MIN)
    )
    vals = df.loc[ok, value_col].to_numpy(dtype=float)
    weights = df.loc[ok, "area"].to_numpy(dtype=float)
    return {
        "n_cells": int(ok.sum()),
        "area_weighted_mean": weighted_mean(vals, weights),
        "area_weighted_median": weighted_median(vals, weights),
    }


def load_tile_table() -> pd.DataFrame:
    tc = read_tilecoord(TILECOORD)
    idx0 = np.asarray(tc["i_indg"], dtype=np.int64) + np.asarray(tc["j_indg"], dtype=np.int64) * M36_NX
    tile = pd.DataFrame({
        "idx0": idx0,
        "lon": np.asarray(tc["com_lon"], dtype=float),
        "lat": np.asarray(tc["com_lat"], dtype=float),
        "area": np.asarray(tc["area"], dtype=float),
    })
    tile = tile[(tile["lat"] >= MAP_LAT_MIN) & np.isfinite(tile["area"]) & (tile["area"] > 0)].copy()
    if tile["idx0"].duplicated().any():
        raise ValueError("tilecoord idx0 values are not unique")
    return tile


tile = load_tile_table()
print(f"Loaded {len(tile):,} M36 land tiles for maps/area-weighting; min lat = {tile['lat'].min():.2f}")


def attach_tile(df: pd.DataFrame) -> pd.DataFrame:
    return df.merge(tile, on="idx0", how="inner")


def load_npz(path: Path):
    if not path.exists():
        raise FileNotFoundError(path)
    return np.load(path, allow_pickle=True)


def sparse_frame(path: Path, value_name: str, values: np.ndarray) -> pd.DataFrame:
    z = load_npz(path)
    return pd.DataFrame({"idx0": np.asarray(z["idx0"], dtype=np.int64), value_name: np.asarray(values, dtype=float)})


def iv_file(sensor: str, run: str) -> Path:
    matches = sorted((BUNDLE / "step4_iv" / sensor / run).glob("*.npz"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one IV file for {sensor}/{run}, found {len(matches)}")
    return matches[0]


def load_iv(sensor: str, run: str, field: str = "R_ivs_mod") -> pd.DataFrame:
    path = iv_file(sensor, run)
    z = load_npz(path)
    if field == "R_ivs_mod":
        r2 = np.asarray(z["R2_ivs_mod"], dtype=float)
        values = np.sqrt(np.clip(r2, 0.0, 1.0))
        values[(~np.isfinite(r2)) | (r2 < 0.0) | (r2 > 1.0)] = np.nan
    else:
        values = np.asarray(z[field], dtype=float)
    df = pd.DataFrame({
        "idx0": np.asarray(z["idx0"], dtype=np.int64),
        field: values,
        "N_sm": np.asarray(z["N_sm"], dtype=float),
    })
    return attach_tile(df)


def sparse_delta(left: pd.DataFrame, right: pd.DataFrame, value_col: str, out_col: str) -> pd.DataFrame:
    both = left[["idx0", value_col]].merge(right[["idx0", value_col]], on="idx0", suffixes=("_left", "_right"))
    both[out_col] = both[f"{value_col}_right"] - both[f"{value_col}_left"]
    return attach_tile(both[["idx0", out_col]])


def tc_file(combo: str, run: str) -> Path:
    matches = sorted((BUNDLE / "step4_tc" / combo / run).glob("*.npz"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one TC file for {combo}/{run}, found {len(matches)}")
    return matches[0]


def load_tc(combo: str, run: str) -> pd.DataFrame:
    path = tc_file(combo, run)
    z = load_npz(path)
    variance = np.asarray(z["variance"], dtype=float)[:, 1]
    sigma2 = np.asarray(z["sigma2"], dtype=float)[:, 1]
    fmse = sigma2 / variance
    fmse[(~np.isfinite(fmse)) | (fmse < 0) | (fmse > 1)] = np.nan
    df = pd.DataFrame({
        "idx0": np.asarray(z["idx0"], dtype=np.int64),
        "fmse_model": fmse,
        "R2_TC_model": np.asarray(z["R2_TC"], dtype=float)[:, 1],
        "N_sm": np.asarray(z["N_sm"], dtype=float),
    })
    return attach_tile(df)


def setup_map(ax, title: str | None = None):
    if HAS_CARTOPY:
        ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, facecolor=LAND_FACE, edgecolor="none", zorder=0)
        ax.coastlines(linewidth=0.35, color="0.25", zorder=2)
    else:
        ax.set_facecolor(LAND_FACE)
        ax.set_xlim(MAP_EXTENT[0], MAP_EXTENT[1])
        ax.set_ylim(MAP_EXTENT[2], MAP_EXTENT[3])
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
    if title:
        ax.set_title(title)


def _panel_label_text(index: int) -> str:
    letters = []
    index = int(index)
    while True:
        letters.append(chr(ord("a") + index % 26))
        index = index // 26 - 1
        if index < 0:
            break
    return f"({''.join(reversed(letters))})"


def add_panel_labels(axes, *, x: float = 0.02, y: float = 0.98) -> None:
    for i, ax in enumerate(np.ravel(np.asarray(axes, dtype=object))):
        if ax is None or not ax.get_visible():
            continue
        ax.text(
            x, y, _panel_label_text(i),
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=10, fontweight="bold",
            bbox=dict(boxstyle="square,pad=0.15", facecolor="white", edgecolor="none", alpha=0.78),
            zorder=20,
        )


def segmented_cmap_norm(cmap_name: str, vlim: float, n_bins: int = 12, neutral_fraction: float = ZERO_NEUTRAL_FRACTION):
    side_bins = max(int(n_bins) // 2, 1)
    color_bins = side_bins * 2
    neutral = max(float(vlim) * float(neutral_fraction), np.finfo(float).eps)
    neutral = min(neutral, float(vlim) * 0.5)
    neg_bounds = np.linspace(-vlim, -neutral, side_bins + 1)
    pos_bounds = np.linspace(neutral, vlim, side_bins + 1)
    bounds = np.r_[neg_bounds, pos_bounds]
    base = plt.get_cmap(cmap_name, color_bins)
    base_colors = base(np.linspace(0, 1, color_bins))
    colors = np.vstack([
        base_colors[:side_bins],
        np.array([[1.0, 1.0, 1.0, 1.0]]),
        base_colors[side_bins:],
    ])
    cmap = ListedColormap(colors, name=f"{cmap_name}_segmented_zero")
    norm = BoundaryNorm(bounds, cmap.N, clip=True)
    return cmap, norm


def set_diverging_colorbar_ticks(cb, vlim: float, fmt: str = "{:.2f}") -> None:
    ticks = np.linspace(-vlim, vlim, 5)
    cb.set_ticks(ticks)
    cb.ax.set_xticklabels([fmt.format(t) for t in ticks])


def scatter_map(ax, df: pd.DataFrame, value_col: str, *, title: str, cmap="RdBu_r", vlim=0.2, s=1.0):
    plot = df[np.isfinite(df[value_col]) & (df["lat"] >= MAP_LAT_MIN)].copy()
    cmap_obj, norm = segmented_cmap_norm(cmap, vlim)
    kwargs = dict(c=plot[value_col], s=s, cmap=cmap_obj, norm=norm, linewidths=0, rasterized=True)
    if HAS_CARTOPY:
        sc = ax.scatter(plot["lon"], plot["lat"], transform=ccrs.PlateCarree(), **kwargs)
    else:
        sc = ax.scatter(plot["lon"], plot["lat"], **kwargs)
    summary = area_summary(plot, value_col)
    setup_map(ax, f"{title}\nmean={summary['area_weighted_mean']:.3f}; n={summary['n_cells']:,}")
    return sc




## Summary Tables



In [ ]:
iv_rows = []
for sensor in SENSOR_ORDER:
    sensor_dir = BUNDLE / "step4_iv" / sensor
    if not sensor_dir.exists():
        continue
    for run in RUN_ORDER:
        if not (sensor_dir / run).exists():
            continue
        df = load_iv(sensor, run)
        summary = area_summary(df, "R_ivs_mod")
        iv_rows.append({
            "sensor": sensor,
            "sensor_label": SENSOR_LABELS.get(sensor, sensor),
            "run": run,
            "run_label": RUN_LABELS.get(run, run),
            **summary,
        })
iv_summary = pd.DataFrame(iv_rows)
iv_summary.to_csv(OUT / "iv_area_weighted_summary.csv", index=False)
display(iv_summary)

tc_rows = []
for combo_dir in sorted((BUNDLE / "step4_tc").iterdir()):
    if not combo_dir.is_dir():
        continue
    combo = combo_dir.name
    for run_dir in sorted(combo_dir.iterdir()):
        if not run_dir.is_dir():
            continue
        run = run_dir.name
        df = load_tc(combo, run)
        fmse_summary = area_summary(df, "fmse_model")
        r2_summary = area_summary(df, "R2_TC_model")
        tc_rows.append({
            "combo": combo,
            "combo_label": TC_COMBO_LABELS.get(combo, combo),
            "run": run,
            "run_label": RUN_LABELS.get(run, run),
            "fmse_aw_mean": fmse_summary["area_weighted_mean"],
            "fmse_aw_median": fmse_summary["area_weighted_median"],
            "r2_aw_mean": r2_summary["area_weighted_mean"],
            "r2_aw_median": r2_summary["area_weighted_median"],
            "n_cells": fmse_summary["n_cells"],
        })
tc_summary = pd.DataFrame(tc_rows)
tc_summary.to_csv(OUT / "tc_area_weighted_summary.csv", index=False)
display(tc_summary)




## Figure 1: IV Global Skill Scorecard

Mean IV model R by instrument-variable sensor and run. Higher bars mean better agreement with the IV sensor.


In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.6))
plot_df = iv_summary[iv_summary["sensor"].isin(SENSOR_ORDER)].copy()
x = np.arange(len(SENSOR_ORDER))
bar_w = 0.18
colors = {"OL": "0.55", "DA_legacy": "#0072B2", "DA_H121": "#D55E00", "DA_SMAP_comb_fp_scaled": "#009E73"}
for i, run in enumerate(RUN_ORDER):
    sub = plot_df[plot_df["run"] == run].set_index("sensor").reindex(SENSOR_ORDER)
    ax.bar(x + (i - 1.5) * bar_w, sub["area_weighted_mean"], width=bar_w, label=RUN_LABELS[run], color=colors[run])
ax.set_xticks(x)
ax.set_xticklabels([SENSOR_LABELS[s] for s in SENSOR_ORDER], rotation=20, ha="right")
ax.set_ylabel("Mean IV model R")
ax.set_ylim(0, max(0.75, np.nanmax(plot_df["area_weighted_mean"]) * 1.10))
ax.grid(axis="y", color="0.88", linewidth=0.7)
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.18), frameon=False)
ax.set_title("Instrument variable model skill by IV sensor")
add_panel_labels([ax])
fig.subplots_adjust(top=0.80, bottom=0.22)
save_fig(fig, "fig01_iv_global_skill_scorecard")
plt.close(fig)




## Figure 2: IV Delta Maps Versus OL

Maps show DA minus OL IV model R for instrument-variable sensors. Red/positive means DA improves skill; blue/negative means DA is worse than OL. White marks near-zero differences.


In [ ]:
map_sensors = ["smap_l3", "smosic", "cygnss_l3"]
map_runs = ["DA_legacy", "DA_H121"]
fig, axes = plt.subplots(
    len(map_sensors), len(map_runs), figsize=(10.8, 9.0),
    subplot_kw={"projection": ccrs.Robinson()} if HAS_CARTOPY else {},
)
for r, sensor in enumerate(map_sensors):
    ol = load_iv(sensor, "OL")
    for c, run in enumerate(map_runs):
        da = load_iv(sensor, run)
        delta = sparse_delta(ol, da, "R_ivs_mod", "delta_R_ivs_mod")
        sc = scatter_map(
            axes[r, c], delta, "delta_R_ivs_mod",
            title=f"{SENSOR_LABELS[sensor]}: {RUN_LABELS[run]} - OL",
            vlim=0.35, s=0.8,
        )
cb = fig.colorbar(sc, ax=axes.ravel().tolist(), orientation="horizontal", fraction=0.035, pad=0.04, label="Delta IV model R")
set_diverging_colorbar_ticks(cb, 0.35)
fig.suptitle("IV skill changes relative to OL (latitude >= -60 deg)", y=0.98, fontsize=12)
add_panel_labels(axes)
save_fig(fig, "fig02_iv_independent_sensor_delta_maps")
plt.close(fig)




## Figure 3: IV H121 Minus Legacy Maps

Maps show DA_H121 minus DA_legacy IV model R. Red/positive favors H121; blue/negative favors legacy. White marks near-zero differences.


In [ ]:
compare_sensors = ["smap_l3", "smosic", "cygnss_l3"]
fig, axes = plt.subplots(
    1, len(compare_sensors), figsize=(12.0, 3.8),
    subplot_kw={"projection": ccrs.Robinson()} if HAS_CARTOPY else {},
)
axes = np.ravel(axes)
for ax, sensor in zip(axes, compare_sensors):
    legacy = load_iv(sensor, "DA_legacy")
    h121 = load_iv(sensor, "DA_H121")
    delta = sparse_delta(legacy, h121, "R_ivs_mod", "delta_R_ivs_mod")
    sc = scatter_map(ax, delta, "delta_R_ivs_mod", title=f"{SENSOR_LABELS[sensor]}: H121 - legacy", vlim=0.25, s=0.8)
cb = fig.colorbar(sc, ax=axes.tolist(), orientation="horizontal", fraction=0.06, pad=0.08, label="Delta IV model R")
set_diverging_colorbar_ticks(cb, 0.25)
fig.suptitle("Direct H121-minus-legacy IV skill comparison", y=0.96, fontsize=12)
add_panel_labels(axes)
save_fig(fig, "fig03_iv_h121_minus_legacy_maps")
plt.close(fig)




## Figure 4: TC Model fMSE Scorecard

Mean TC model fractional MSE by triplet and run. Lower bars mean lower model error.


In [ ]:
combo_order = ["smap_l3__model__smosic", "cygnss_l3__model__smosic", "smosic__model__ascat_h121"]
fig, ax = plt.subplots(figsize=(10.4, 4.8))
x = np.arange(len(combo_order))
bar_w = 0.18
for i, run in enumerate(RUN_ORDER):
    vals = []
    for combo in combo_order:
        row = tc_summary[(tc_summary["combo"] == combo) & (tc_summary["run"] == run)]
        vals.append(row["fmse_aw_mean"].iloc[0] if len(row) else np.nan)
    ax.bar(x + (i - 1.5) * bar_w, vals, width=bar_w, label=RUN_LABELS[run], color=colors[run])
ax.set_xticks(x)
ax.set_xticklabels([TC_COMBO_LABELS[c].replace(" + ", "\n+ ") for c in combo_order])
ax.set_ylabel("Mean model fMSE")
ax.set_ylim(0, 0.85)
ax.grid(axis="y", color="0.88", linewidth=0.7)
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.18), frameon=False)
ax.set_title("Triple Collocation model fractional error variance")
add_panel_labels([ax])
fig.subplots_adjust(top=0.80, bottom=0.25)
save_fig(fig, "fig04_tc_model_fmse_scorecard")
plt.close(fig)




## Figure 5: TC fMSE Improvement Maps Versus OL

Maps show OL minus DA TC model fMSE. Red/positive means DA lowers model error; blue/negative means DA increases error. White marks near-zero differences.


In [ ]:
tc_map_combos = ["smap_l3__model__smosic", "cygnss_l3__model__smosic"]
tc_map_runs = ["DA_legacy", "DA_H121"]
fig, axes = plt.subplots(
    len(tc_map_combos), len(tc_map_runs), figsize=(10.8, 6.5),
    subplot_kw={"projection": ccrs.Robinson()} if HAS_CARTOPY else {},
)
for r, combo in enumerate(tc_map_combos):
    ol = load_tc(combo, "OL")
    for c, run in enumerate(tc_map_runs):
        da = load_tc(combo, run)
        both = ol[["idx0", "fmse_model"]].merge(da[["idx0", "fmse_model"]], on="idx0", suffixes=("_OL", "_DA"))
        both["fmse_improvement"] = both["fmse_model_OL"] - both["fmse_model_DA"]
        delta = attach_tile(both[["idx0", "fmse_improvement"]])
        sc = scatter_map(ax=axes[r, c], df=delta, value_col="fmse_improvement", title=f"{TC_COMBO_LABELS[combo]}\n{RUN_LABELS[run]}: OL - DA", vlim=0.25, s=0.8)
cb = fig.colorbar(sc, ax=axes.ravel().tolist(), orientation="horizontal", fraction=0.045, pad=0.05, label="Model fMSE improvement (OL - DA)")
set_diverging_colorbar_ticks(cb, 0.25)
fig.suptitle("TC fMSE reductions relative to OL", y=0.98, fontsize=12)
add_panel_labels(axes)
save_fig(fig, "fig05_tc_fmse_improvement_maps")
plt.close(fig)




## Figure 6: TC H121 Advantage Over Legacy

Maps show legacy minus H121 TC model fMSE. Red/positive means H121 has lower error; blue/negative means legacy has lower error. White marks near-zero differences.


In [ ]:
fig, axes = plt.subplots(
    1, len(tc_map_combos), figsize=(10.8, 3.7),
    subplot_kw={"projection": ccrs.Robinson()} if HAS_CARTOPY else {},
)
for ax, combo in zip(np.ravel(axes), tc_map_combos):
    legacy = load_tc(combo, "DA_legacy")
    h121 = load_tc(combo, "DA_H121")
    both = legacy[["idx0", "fmse_model"]].merge(h121[["idx0", "fmse_model"]], on="idx0", suffixes=("_legacy", "_h121"))
    both["h121_advantage"] = both["fmse_model_legacy"] - both["fmse_model_h121"]
    delta = attach_tile(both[["idx0", "h121_advantage"]])
    sc = scatter_map(ax, delta, "h121_advantage", title=TC_COMBO_LABELS[combo], vlim=0.15, s=0.8)
cb = fig.colorbar(sc, ax=np.ravel(axes).tolist(), orientation="horizontal", fraction=0.06, pad=0.08, label="Model fMSE advantage (legacy - H121; positive = H121 lower error)")
set_diverging_colorbar_ticks(cb, 0.15)
fig.suptitle("Direct H121-minus-legacy TC comparison", y=0.98, fontsize=12)
add_panel_labels(axes)
save_fig(fig, "fig06_tc_h121_minus_legacy_maps")
plt.close(fig)




## Notes



The ASCAT-family IV panels are useful diagnostics but should not be treated as fully independent evidence for a run that assimilates the same observation family. The TC triplets intentionally omit circular run/triplet combinations, following the bundle README.

